# Umbral y métricas

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Tipos de ataque

`NORMAL_CAT` y `ATTACK_TYPES` guardan los valores de `attack_cat`: el tráfico normal y los 9 tipos de ataque, en el orden del contrato. Las métricas del tipo de ataque los usan para listar siempre los 9 tipos, en el mismo orden.

Comprobamos que coinciden exactamente con los valores de `attack_cat` del archivo de entrenamiento y del de prueba (sin faltar ni sobrar ninguno), y que en los dos archivos `label` = 1 equivale a que `attack_cat` sea uno de los 9 tipos de ataque. Aquí la prueba solo se abre para leer sus categorías.

In [ ]:
from threat_detector.dataset import ATTACK_CAT, ATTACK_TYPES, NORMAL_CAT, TARGET, load_test, load_train

expected = {NORMAL_CAT, *ATTACK_TYPES}
print(f"Tipos de ataque: {len(ATTACK_TYPES)}; ¿sin repetidos?: {len(set(ATTACK_TYPES)) == len(ATTACK_TYPES)}")

for name, flows in (("Entrenamiento", load_train()), ("Prueba", load_test())):
    found = set(flows[ATTACK_CAT].unique())
    label_matches = (flows[TARGET] == 1).equals(flows[ATTACK_CAT].isin(ATTACK_TYPES))
    print(f"{name}: ¿mismos valores de attack_cat?: {found == expected} "
          f"(faltan: {sorted(expected - found)}, sobran: {sorted(found - expected)}); "
          f"¿label = 1 si y solo si es un tipo de ataque?: {label_matches}")

## 2. Alerta y severidad

`is_alert` es la única regla de alerta del proyecto: un flujo es alerta si su probabilidad de ataque es **mayor o igual** que el umbral del modelo. `severity` la usa como primer paso y después compara con los cortes de `config.py` (`SEVERITY_CRITICAL`, `SEVERITY_HIGH`, `SEVERITY_MEDIUM`). Las dos están en `artifacts.py`, para que la app y `ml/` usen exactamente la misma regla.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos probabilidades y umbrales escritos a mano, elegidos justo en los límites. Las funciones se aplicarán a las probabilidades reales de cada modelo en la etapa de modelos.

**Resultado esperado:**
- Con umbral 0,31: una probabilidad justo en el umbral es `"low"`, y justo en cada corte sube de nivel (0,60 → `"medium"`, 0,75 → `"high"`, 0,90 → `"critical"`); un poco por debajo de cada uno se queda en el nivel anterior.
- Con umbral 0,70 (mayor que 0,60) ninguna probabilidad da `"low"`: el nivel queda vacío, y es correcto.
- `is_alert` es verdadero si y solo si la severidad no es `"normal"`; con muchos flujos devuelve un valor por flujo, y con uno solo devuelve un `bool` de Python que se puede escribir en JSON.

In [ ]:
import json

import numpy as np
import pandas as pd

from threat_detector.artifacts import is_alert, severity
from threat_detector.config import SEVERITY_CRITICAL, SEVERITY_HIGH, SEVERITY_MEDIUM

# Probabilidades inventadas con la severidad que debe salir, para dos umbrales inventados.
cases = {
    0.31: [(0.30, "normal"), (0.31, "low"), (0.5999, "low"), (SEVERITY_MEDIUM, "medium"),
           (0.7499, "medium"), (SEVERITY_HIGH, "high"), (0.8999, "high"),
           (SEVERITY_CRITICAL, "critical"), (1.0, "critical")],
    0.70: [(0.65, "normal"), (0.70, "medium"), (0.7499, "medium"), (0.80, "high"), (0.95, "critical")],
}
rows = [
    {"umbral": threshold, "probabilidad": p, "esperada": expected, "obtenida": severity(p, threshold),
     "is_alert": is_alert(p, threshold)}
    for threshold, pairs in cases.items()
    for p, expected in pairs
]
results = pd.DataFrame(rows)
results["¿coincide?"] = results["esperada"] == results["obtenida"]

print(f"¿Todas las severidades coinciden?: {results['¿coincide?'].all()}")
print(f"¿Algún 'low' con umbral 0,70?: {'low' in results.loc[results['umbral'] == 0.70, 'obtenida'].tolist()}")
print(f"¿is_alert si y solo si la severidad no es 'normal'?: {(results['is_alert'] == (results['obtenida'] != 'normal')).all()}")

many = is_alert(np.array([0.10, 0.31, 0.95]), 0.31)
one = is_alert(np.float64(0.5), 0.31)
print(f"is_alert con 3 flujos: {many}; con un flujo: {one!r} (tipo {type(one).__name__})")
print(f"Un flujo en JSON: {json.dumps({'is_alert': one, 'severity': severity(0.5, 0.31)})}")

results

## 3. Tasas de detección y de falsas alarmas

`detection_rate` es la parte de los ataques reales (`label` = 1) que dio alerta, y `false_alarm_rate` la parte de los flujos normales reales (`label` = 0) que dio alerta. Están en `artifacts.py` porque las usan la app (en los indicadores de un CSV con etiquetas) y `ml/evaluation.py` (en validación y prueba). Si no hay flujos de la clase que se divide, devuelven `None` (`null` en el JSON), como dice el contrato para el CSV del usuario.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos etiquetas y alertas escritas a mano.

**Resultado esperado:**
- 4 ataques reales, 3 con alerta, y 5 normales, 1 con alerta: detección = 3/4 = 0,75 y falsas alarmas = 1/5 = 0,2.
- Un archivo con solo ataques: falsas alarmas = `None`. Un archivo con solo normales: detección = `None`.
- Los resultados son números de Python y se pueden escribir en JSON.

In [ ]:
from threat_detector.artifacts import detection_rate, false_alarm_rate

# Etiquetas reales y alertas inventadas, con el resultado esperado.
rate_cases = [
    ("4 ataques (3 alertas) y 5 normales (1 alerta)", [1, 1, 1, 1, 0, 0, 0, 0, 0],
     [True, True, True, False, True, False, False, False, False], 0.75, 0.2),
    ("Solo ataques (1 de 2 con alerta)", [1, 1], [True, False], 0.5, None),
    ("Solo normales (ninguno con alerta)", [0, 0, 0], [False, False, False], None, 0.0),
]
for name, y_true, alerts, expected_detection, expected_false_alarm in rate_cases:
    detection = detection_rate(y_true, alerts)
    false_alarm = false_alarm_rate(y_true, alerts)
    ok = detection == expected_detection and false_alarm == expected_false_alarm
    print(f"{name}: detección = {detection}, falsas alarmas = {false_alarm}; ¿lo esperado?: {ok}; "
          f"JSON: {json.dumps({'detection_rate': detection, 'false_alarm_rate': false_alarm})}")

## 4. Umbral

`compute_threshold` calcula el umbral de alerta de un modelo con las predicciones de validación: el umbral **más alto** que deja al menos `TARGET_RECALL` (95 %) de los ataques reales como alerta. Ordena las probabilidades de los ataques reales de mayor a menor y toma la del puesto *k*, donde *k* es el número de ataques que deben quedar como alerta (95 % redondeado hacia arriba). Los flujos normales no cambian el umbral: las falsas alarmas son consecuencia, no objetivo.

> **Datos inventados.** Como aún no hay modelos entrenados, en los casos 1, 3, 4 y 5 usamos probabilidades escritas a mano (redondeadas a 2 decimales, para que el punto flotante no cambie los valores esperados). La función se aplicará a las probabilidades de validación de cada modelo en la etapa de modelos.

**Resultado esperado:**
1. **20 ataques** con probabilidades 0,05; 0,10; …; 1,00 y 10 normales: hacen falta 19 ataques, así que el umbral es **0,10** (la 19.ª más alta). Con 0,10 la detección es 19/20 = 0,95; con 0,1001 baja a 18/20 = 0,90; el ataque con probabilidad exactamente 0,10 es alerta.
2. **El cálculo es exacto:** el número de ataques que deben quedar como alerta se calcula como `ceil(TARGET_RECALL × ataques)`. Como `TARGET_RECALL` es un número decimal guardado en punto flotante, comprobamos para todos los tamaños de 1 a 50.000 ataques (la validación tiene unos 24.000) que ese cálculo da lo mismo que el cálculo exacto con fracciones. Debe coincidir en todos. Esta prueba confirma que el cálculo simple es exacto **para el valor configurado** de `TARGET_RECALL`; si `TARGET_RECALL` cambia en `config.py`, hay que volver a correrla.
3. **Empate en el umbral:** si dos ataques tienen 0,10, el umbral sigue siendo 0,10 y la detección sube a 20/20 = 1,0. Lo calculamos con dos grupos distintos de normales (probabilidades muy bajas y muy altas): el umbral es el mismo, porque los normales no lo cambian.
4. **7 ataques:** 0,95 × 7 = 6,65, así que hacen falta los 7: el umbral es la probabilidad más baja de los ataques (0,40) y la detección es 1,0. También con dos grupos distintos de normales, y el umbral no cambia.
5. **Errores:** etiquetas de una sola clase (la función exige flujos normales y ataques reales, igual que `threshold_rule`, que se calcula con los mismos datos de validación) y distinta cantidad de etiquetas y probabilidades. Los dos dan un mensaje en español.

In [ ]:
import math
from fractions import Fraction

from threat_detector.artifacts import detection_rate
from threat_detector.config import TARGET_RECALL
from threat_detector.ml.evaluation import compute_threshold

# Caso 1: 20 ataques con probabilidades 0,05 a 1,00 (redondeadas a 2 decimales) y 10 normales.
attack_probs = np.round(np.arange(1, 21) * 0.05, 2)
normal_probs = np.round(np.arange(10) * 0.03, 2)
y_true = np.array([1] * 20 + [0] * 10)
probabilities = np.concatenate([attack_probs, normal_probs])

threshold = compute_threshold(y_true, probabilities)
print(f"Caso 1: umbral = {threshold} (esperado 0.1)")
print(f"    detección con el umbral: {detection_rate(y_true, is_alert(probabilities, threshold))} (esperada 0.95)")
print(f"    detección con 0.1001: {detection_rate(y_true, is_alert(probabilities, 0.1001))} (esperada 0.9)")
print(f"    ¿el ataque con probabilidad 0.10 es alerta?: {is_alert(0.10, threshold)}")

# Caso 2: el cálculo con punto flotante da lo mismo que el cálculo exacto con fracciones.
# Fraction(str(...)) toma el valor decimal escrito en config.py (por ejemplo 0.95 = 19/20).
exact_recall = Fraction(str(TARGET_RECALL))
mismatches = [n for n in range(1, 50_001) if math.ceil(TARGET_RECALL * n) != math.ceil(exact_recall * n)]
print(f"\nCaso 2: TARGET_RECALL = {TARGET_RECALL} = {exact_recall}; "
      f"tamaños de 1 a 50.000 ataques donde el cálculo simple no es exacto: {len(mismatches)} (esperado 0)")

# Casos 3 y 4: dos grupos de normales inventados, con probabilidades muy bajas y muy altas.
normal_groups = {"normales bajos": [0.01, 0.02, 0.03], "normales altos": [0.97, 0.98, 0.99]}

# Caso 3: dos ataques empatados en 0,10.
tied = [0.10, 0.10] + [round(0.05 * i, 2) for i in range(3, 21)]
print()
for group, normals in normal_groups.items():
    labels = [1] * 20 + [0] * len(normals)
    probs = tied + normals
    tied_threshold = compute_threshold(labels, probs)
    print(f"Caso 3 ({group}): umbral = {tied_threshold} (esperado 0.1); "
          f"detección = {detection_rate(labels, is_alert(probs, tied_threshold))} (esperada 1.0)")

# Caso 4: 7 ataques, hacen falta los 7.
seven = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
print()
for group, normals in normal_groups.items():
    labels = [1] * 7 + [0] * len(normals)
    probs = seven + normals
    seven_threshold = compute_threshold(labels, probs)
    print(f"Caso 4 ({group}): umbral = {seven_threshold} (esperado 0.4); "
          f"detección = {detection_rate(labels, is_alert(probs, seven_threshold))} (esperada 1.0)")

# Caso 5: errores.
print()
for name, bad_labels, bad_probs in (
    ("Solo normales", [0, 0, 0], [0.1, 0.2, 0.3]),
    ("Distinta cantidad", [1, 0, 1], [0.1, 0.2]),
):
    try:
        compute_threshold(bad_labels, bad_probs)
    except ValueError as error:
        print(f"Caso 5, {name}: {error}")

## 5. Regla del umbral

`threshold_rule` arma el objeto `threshold_rule` de `model.json` (contrato, sección 11): qué detección mínima se exigió, que el umbral se calculó con la validación, cuántos flujos de validación se usaron y qué detección y falsas alarmas logró ese umbral. Las dos tasas salen de `detection_rate` y `false_alarm_rate` de `artifacts.py`, sin calcularlas otra vez.

Antes de calcular, comprueba que las etiquetas y las probabilidades traen los mismos flujos y que hay flujos normales y ataques reales; si no, lanza un error en español.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos los mismos datos escritos a mano del caso 1 de la sección 4: 20 ataques y 10 normales, con umbral 0,10.

**Resultado esperado:**
- `target_recall` = `TARGET_RECALL` (0,95) y `computed_on` = `"validation"`.
- `validation_flows` = 30.
- `validation_detection_rate` = 19/20 = 0,95.
- `validation_false_alarm_rate` = 0,6: los normales tienen probabilidades 0; 0,03; …; 0,27, y 6 de los 10 son ≥ 0,10.
- Las claves son exactamente las del contrato y el resultado se puede escribir en JSON.
- Con etiquetas de una sola clase, o con distinta cantidad de etiquetas y probabilidades, un error en español.

In [ ]:
from threat_detector.ml.evaluation import threshold_rule

rule = threshold_rule(y_true, probabilities, threshold)
expected_rule = {
    "target_recall": TARGET_RECALL,
    "computed_on": "validation",
    "validation_flows": 30,
    "validation_detection_rate": 0.95,
    "validation_false_alarm_rate": 0.6,
}
print(f"threshold_rule: {json.dumps(rule)}")
print(f"¿Es lo esperado?: {rule == expected_rule}")
print(f"¿Claves en el orden del contrato?: {list(rule) == list(expected_rule)}")

for name, labels, probs in (
    ("Solo ataques", [1, 1, 1], [0.2, 0.5, 0.9]),
    ("Distinta cantidad", [1, 0, 1], [0.2, 0.5]),
):
    try:
        threshold_rule(labels, probs, 0.3)
    except ValueError as error:
        print(f"{name}: {error}")

## 6. Métricas de detección

`detection_metrics` arma el objeto `detection` de las métricas de prueba (contrato, sección 7). Compara las alertas (`is_alert` con el umbral del modelo) con las etiquetas reales: un ataque real con alerta es un verdadero positivo (`tp`), un normal con alerta un falso positivo (`fp`), un normal sin alerta un verdadero negativo (`tn`) y un ataque sin alerta un falso negativo (`fn`). La detección y las falsas alarmas salen de `artifacts.py`; `pr_auc` y `roc_auc` no usan el umbral, sino las probabilidades.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos probabilidades escritas a mano, elegidas para poder calcular cada métrica a mano. La función se aplicará a las probabilidades reales de la prueba de cada modelo en la etapa de modelos.

**Resultado esperado:**
1. **Caso general:** 5 ataques con probabilidades 0,90; 0,80; 0,70; 0,60; 0,20 y 5 normales con 0,65; 0,30; 0,20; 0,10; 0,05, con umbral 0,50.
   - `tp` = 4 (0,90 a 0,60), `fn` = 1 (0,20), `fp` = 1 (0,65), `tn` = 4.
   - Detección = 4/5 = 0,8; falsas alarmas = 1/5 = 0,2; precisión = 4/5 = 0,8; `f1` = 2·4 / (2·4 + 1 + 1) = 0,8; `accuracy` = 8/10 = 0,8.
   - `roc_auc` = **0,86**: de los 25 pares ataque–normal, el ataque tiene la probabilidad más alta en 21 y hay un empate (0,20 y 0,20) que cuenta medio: 21,5 / 25.
   - `pr_auc` = **0,885**: recorriendo las probabilidades de mayor a menor, cada vez que la detección sube 0,2 se suma 0,2 × la precisión en ese punto: 0,2·1 + 0,2·1 + 0,2·1 + 0,2·0,8 + 0,2·0,625.
2. **Separación perfecta** (todos los ataques por encima de todos los normales): `pr_auc` = `roc_auc` = 1,0.
3. **Sin ninguna alerta** (umbral 0,99): `precision` = `None` (`null` en el JSON), `f1` = 0, detección = 0 y falsas alarmas = 0.
4. **Errores:** etiquetas de una sola clase, o distinta cantidad de etiquetas y probabilidades.
5. Las claves están en el orden del contrato y el resultado se puede escribir en JSON.

In [ ]:
from threat_detector.ml.evaluation import detection_metrics

# Caso 1: probabilidades inventadas con resultado calculado a mano.
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
probs = [0.90, 0.80, 0.70, 0.60, 0.20, 0.65, 0.30, 0.20, 0.10, 0.05]
detection = detection_metrics(labels, probs, 0.50)
expected_detection = {
    "detection_rate": 0.8, "false_alarm_rate": 0.2, "precision": 0.8, "f1": 0.8,
    "pr_auc": 0.885, "roc_auc": 0.86, "accuracy": 0.8,
    "confusion": {"tp": 4, "fp": 1, "tn": 4, "fn": 1},
}
print(f"Caso 1: {json.dumps(detection)}")
close = all(
    detection[key] == value if key == "confusion" else math.isclose(detection[key], value)
    for key, value in expected_detection.items()
)
print(f"    ¿Es lo esperado?: {close}; ¿claves en el orden del contrato?: {list(detection) == list(expected_detection)}")

# Caso 2: separación perfecta.
perfect = detection_metrics([1, 1, 0, 0], [0.9, 0.8, 0.3, 0.1], 0.50)
print(f"\nCaso 2: pr_auc = {perfect['pr_auc']}, roc_auc = {perfect['roc_auc']} (esperados 1.0)")

# Caso 3: ninguna alerta.
no_alerts = detection_metrics(labels, probs, 0.99)
print(f"\nCaso 3: precision = {no_alerts['precision']}, f1 = {no_alerts['f1']}, "
      f"detección = {no_alerts['detection_rate']}, falsas alarmas = {no_alerts['false_alarm_rate']}")
print(f"    En JSON: {json.dumps({'precision': no_alerts['precision'], 'f1': no_alerts['f1']})}")

# Caso 4: errores.
for name, bad_labels, bad_probs in (
    ("Solo normales", [0, 0, 0], [0.2, 0.5, 0.9]),
    ("Distinta cantidad", [1, 0, 1, 0], [0.2, 0.5, 0.9]),
):
    try:
        detection_metrics(bad_labels, bad_probs, 0.50)
    except ValueError as error:
        print(f"\nCaso 4, {name}: {error}")

## 7. Métricas del tipo de ataque

`attack_type_metrics` arma el objeto `attack_type` de las métricas de prueba (contrato, sección 7). Solo evalúa los **ataques reales que dieron alerta**, porque el tipo probable solo existe en las alertas; por eso `evaluated_flows` es igual a `tp` de `detection_metrics`. Compara el tipo real con el predicho entre los 9 tipos de `ATTACK_TYPES` y devuelve, para cada tipo, precisión, recall, `f1` y flujos reales (`support`), el promedio `macro_f1` y la matriz de confusión de 9 × 9 (fila = tipo real, columna = tipo predicho).

Cuando una métrica no se puede calcular es `None` (`null` en el JSON): la precisión de un tipo que nunca se predijo, el recall de un tipo sin flujos reales, y el `f1` de un tipo que no aparece ni en lo real ni en lo predicho. `macro_f1` promedia solo los `f1` que se pueden calcular.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos tipos reales, tipos predichos y alertas escritos a mano, elegidos para poder calcular cada métrica a mano. La función se aplicará a las predicciones reales de la prueba de cada modelo en la etapa de modelos.

| Flujos | Tipo real | Tipo predicho | ¿Alerta? |
|---|---|---|---|
| 3 | Generic | Generic | Sí |
| 1 | Generic | Exploits | Sí |
| 2 | Exploits | Exploits | Sí |
| 1 | Exploits | Generic | Sí |
| 1 | DoS | Exploits | Sí |
| 1 | Fuzzers | — | No (queda fuera: no es alerta) |
| 1 | Normal | Generic | Sí (queda fuera: no es un ataque real) |

**Resultado esperado:**
- `evaluated_flows` = 8, igual a `tp` de `detection_metrics` con las mismas alertas.
- Generic: precisión = 3/4 = 0,75, recall = 3/4 = 0,75, `f1` = 0,75, `support` = 4.
- Exploits: precisión = 2/4 = 0,5, recall = 2/3, `f1` = 2·2 / (3 + 4) = 4/7 ≈ 0,571, `support` = 3.
- DoS (nunca predicho): precisión `None`, recall 0, `f1` 0, `support` = 1.
- Los otros 6 tipos (incluido Fuzzers, porque su único flujo no es alerta, y Worms, que no aparece): todo `None` y `support` = 0.
- `macro_f1` = (0,75 + 4/7 + 0) / 3 ≈ 0,440.
- Matriz de 9 × 9: [Generic][Generic] = 3, [Generic][Exploits] = 1, [Exploits][Exploits] = 2, [Exploits][Generic] = 1, [DoS][Exploits] = 1 y el resto 0.
- Sin ningún flujo evaluado: `evaluated_flows` = 0 y `macro_f1` = `None`.
- Errores en español: un ataque real evaluado con un tipo que no es de los 9, una alerta con tipo probable `"Normal"` y distinta cantidad de flujos.

In [ ]:
from threat_detector.dataset import ATTACK_TYPES
from threat_detector.ml.evaluation import attack_type_metrics

rows = (
    [("Generic", "Generic", True)] * 3 + [("Generic", "Exploits", True)]
    + [("Exploits", "Exploits", True)] * 2 + [("Exploits", "Generic", True)]
    + [("DoS", "Exploits", True), ("Fuzzers", None, False), ("Normal", "Generic", True)]
)
true_cats, predicted_cats, alerts = (list(column) for column in zip(*rows))
attack_type = attack_type_metrics(true_cats, predicted_cats, alerts)

# El mismo caso visto como detección: probabilidad 0,9 si es alerta y 0,1 si no, con umbral 0,5.
type_labels = [int(cat != NORMAL_CAT) for cat in true_cats]
type_detection = detection_metrics(type_labels, [0.9 if a else 0.1 for a in alerts], 0.5)
print(f"evaluated_flows = {attack_type['evaluated_flows']}; tp de detection_metrics = {type_detection['confusion']['tp']} (esperados 8)")

by_type = {c["attack_cat"]: c for c in attack_type["per_class"]}
expected_types = {
    "Generic": {"precision": 0.75, "recall": 0.75, "f1": 0.75, "support": 4},
    "Exploits": {"precision": 0.5, "recall": 2 / 3, "f1": 4 / 7, "support": 3},
    "DoS": {"precision": None, "recall": 0.0, "f1": 0.0, "support": 1},
}
empty = {"precision": None, "recall": None, "f1": None, "support": 0}
types_ok = all(
    {k: v for k, v in by_type[cat].items() if k != "attack_cat"} == expected_types.get(cat, empty)
    for cat in ATTACK_TYPES
)
print(f"¿Métricas de los 9 tipos como se esperaba?: {types_ok}; "
      f"¿los 9 en orden?: {[c['attack_cat'] for c in attack_type['per_class']] == ATTACK_TYPES}")
print(f"macro_f1 = {attack_type['macro_f1']:.4f} (esperado {(0.75 + 4 / 7 + 0) / 3:.4f})")

matrix = pd.DataFrame(attack_type["confusion"]["matrix"], index=ATTACK_TYPES, columns=ATTACK_TYPES)
print(f"Matriz: {matrix.shape}; ¿labels = ATTACK_TYPES?: {attack_type['confusion']['labels'] == ATTACK_TYPES}")

nothing = attack_type_metrics(["Generic", "Normal"], [None, None], [False, False])
print(f"\nSin flujos evaluados: evaluated_flows = {nothing['evaluated_flows']}, macro_f1 = {nothing['macro_f1']}")
print(f"¿Todo se puede escribir en JSON?: {bool(json.dumps(attack_type)) and bool(json.dumps(nothing))}")

print()
for name, bad_true, bad_pred, bad_alerts in (
    ("Tipo real que no es de los 9", ["Generic", "worms"], ["Generic", "Worms"], [True, True]),
    ("Alerta con tipo probable Normal", ["Generic", "DoS"], ["Generic", "Normal"], [True, True]),
    ("Distinta cantidad", ["Generic", "DoS"], ["Generic"], [True, True]),
):
    try:
        attack_type_metrics(bad_true, bad_pred, bad_alerts)
    except ValueError as error:
        print(f"{name}: {error}")

matrix

## 8. Formato del contrato

Las secciones anteriores comprobaron cada función por separado. Aquí comprobamos el conjunto contra **el contrato mismo**: armamos `threshold_rule` y `test_metrics` (`test_flows`, `detection` y `attack_type`, como en `model.json`) y los comparamos con el ejemplo de `model.json` que trae `docs/contracts.md` (sección 11).

Primero leemos todos los ejemplos JSON del contrato. Si alguno no se puede interpretar (por ejemplo, porque tiene comentarios o puntos suspensivos), la sección lo dice e indica cuál es y en qué sección del contrato está.

Después comparamos la **estructura** de forma recursiva: las mismas claves, en el mismo orden, en cada nivel (y una entrada por cada tipo de ataque en `per_class`). Los valores no se comparan, porque los del contrato son inventados. Por último, comprobamos que el resultado se puede escribir en JSON y leerse de vuelta sin cambios, es decir, que no quedan tipos de numpy.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos los resultados con datos escritos a mano de las secciones 5 y 7. `test_flows` es la cantidad de flujos de esos datos; en la etapa de modelos lo pondrá el código de entrenamiento con los flujos de la prueba.

In [ ]:
import re

from threat_detector.config import ROOT

contract_text = (ROOT / "docs" / "contracts.md").read_text(encoding="utf-8")

# Cada ejemplo JSON del contrato, con la sección (encabezado "## ") en la que está.
examples = []
section = None
for match in re.finditer(r"^## ([^\n]+)$|^```json\n(.*?)^```", contract_text, re.MULTILINE | re.DOTALL):
    if match.group(1):
        section = match.group(1)
    else:
        examples.append((section, match.group(2)))

parsed = []
for number, (section, block) in enumerate(examples, start=1):
    try:
        parsed.append((section, json.loads(block)))
    except json.JSONDecodeError as error:
        print(f"No se pudo interpretar el ejemplo JSON n.º {number} (sección «{section}»): {error}")
print(f"Ejemplos JSON del contrato: {len(examples)}; interpretados: {len(parsed)}")

model_examples = [example for _, example in parsed if "test_metrics" in example and "threshold_rule" in example]
if not model_examples:
    raise ValueError("No se encontró en contracts.md el ejemplo de model.json (con test_metrics y threshold_rule).")
model_json = model_examples[0]


def structure(value):
    """Estructura de un valor JSON: claves y orden de los objetos, sin los valores."""
    if isinstance(value, dict):
        return [(key, structure(item)) for key, item in value.items()]
    if isinstance(value, list) and value and isinstance(value[0], dict):
        return [structure(item) for item in value]
    return "valor"


test_metrics = {"test_flows": len(type_labels), "detection": type_detection, "attack_type": attack_type}
for name, ours, theirs in (
    ("threshold_rule", rule, model_json["threshold_rule"]),
    ("test_metrics", test_metrics, model_json["test_metrics"]),
):
    print(f"{name}: ¿misma estructura que el contrato?: {structure(ours) == structure(theirs)}; "
          f"¿JSON de ida y vuelta sin cambios?: {json.loads(json.dumps(ours)) == ours}")

print(f"¿evaluated_flows = tp?: {attack_type['evaluated_flows'] == type_detection['confusion']['tp']}")